# M2 — Historical query retrieval: E04

**Цель.** Проверить, дают ли train interactions дополнительные кандидаты для повторяющихся или похожих запросов без использования `query_id` и без доступа к labels validation-query.

**Критерий решения.** Исторический source принимается только если он даёт измеримую общую пользу на frozen group-disjoint proxy или является строго exact full-context lookup с документированным coverage в финальном benchmark. Решение о финальном смешивании всех sources остаётся за M5–M6.

## План ноутбука

1. **Reproducibility и ClearML** — загружаем live-конфигурацию и фиксируем параметры E04.
2. **Frozen proxy** — воспроизводим M0 split; history строится только из train-групп вне validation.
3. **Exact history** — измеряем, почему exact full-context lookup не оценивается на group-disjoint fold, и его coverage на benchmark queries.
4. **Relaxed history** — проверяем lookup по `query text + category` и по одному query text.
5. **Nearest history** — тестируем char-TFIDF nearest historical query, в том числе только для ранее невиденного текста.
6. **Candidate-budget test** — проверяем, улучшают ли history candidates retained M1 baseline при лимите 50.
7. **Confirmation split** — повторяем выбранную policy на независимом group-disjoint seed.
8. **Results и решение** — логируем aggregate metrics в ClearML и принимаем/reject каждый source.

Следующие разделы кратко объясняют назначение ячеек; код комментирует только нетривиальные ограничения против leakage.

## 1. Reproducibility и live ClearML

Следующие ячейки загружают `.env` до импорта ClearML, требуют live credentials и не печатают секреты. В ClearML уйдут только агрегированные метрики, параметры и runtime.

In [1]:
from __future__ import annotations

import os
import re
import resource
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
np.random.seed(SEED)
REPO_ROOT = Path.cwd()
DATA_DIR = Path(os.environ.get("AVITO_DATA_DIR", "/Users/kite/Downloads/dataset"))
TRAIN_PATH = DATA_DIR / "train.parquet"
BENCHMARK_QUERIES_PATH = DATA_DIR / "benchmark_queries.parquet"
BENCHMARK_ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"

TOP_K = 200
FINAL_K = 50
METRIC_KS = (1, 5, 10, 20, 50, 200)
BM25_K1, BM25_B = 1.5, 0.75
CHAR_MAX_FEATURES = 200_000
HISTORY_QUOTAS = (1, 3, 5, 10, 20)
NOTEBOOK_STARTED = time.perf_counter()


def load_dotenv(path: Path) -> None:
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


assert REPO_ROOT.joinpath(".env").exists(), "Fill .env with live ClearML credentials."
assert all(path.exists() for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH))
load_dotenv(REPO_ROOT / ".env")
missing = [key for key in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY") if not os.environ.get(key)]
if missing:
    raise RuntimeError(f"Missing ClearML settings: {', '.join(missing)}")
if os.environ.get("CLEARML_OFFLINE_MODE", "").lower() in {"1", "true", "yes"}:
    raise RuntimeError("M2 requires live ClearML; offline mode is disabled.")

print({"seed": SEED, "top_k": TOP_K, "final_k": FINAL_K, "clearml_credentials_present": True})

{'seed': 42, 'top_k': 200, 'final_k': 50, 'clearml_credentials_present': True}


In [2]:
from clearml import Task

clearml_task = Task.init(
    project_name="avito-retrieval",
    task_name="E04__query_history__s42",
    reuse_last_task_id=False,
    auto_connect_frameworks=False,
)
clearml_task.connect(
    {
        "stage": "M2_historical_signal",
        "validation_protocol": "benchmark_aligned_proxy_v1",
        "seed": SEED,
        "top_k": TOP_K,
        "final_k": FINAL_K,
        "sources": ["exact_full_context", "text_category", "text", "char_tfidf_nearest_unseen"],
        "category_rule": "item_category_id == search_category; full-corpus fallback without matching partition",
        "history_quota_grid": list(HISTORY_QUOTAS),
        "confirmation_seed": 314,
    },
    name="config",
)
clearml_logger = clearml_task.get_logger()
print({"clearml_task_id": clearml_task.id, "offline_mode": False})

ClearML Task: created new task id=494d2e7e5f274c3c99b40255ec1900a6


2026-09-18 15:13:18,570 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []


ClearML results page: https://app.clear.ml/projects/588424e922a44a95aa934ad17ca57931/tasks/494d2e7e5f274c3c99b40255ec1900a6/output/log
2026-09-18 15:13:19,190 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


{'clearml_task_id': '494d2e7e5f274c3c99b40255ec1900a6', 'offline_mode': False}


## 2. Frozen M0 proxy и допустимая история

Следующие ячейки воспроизводят M0 exactly. Для validation history исключает все строки тех же `query_group`, поэтому повторный запрос из validation не может вернуть собственные labels через train lookup.

In [3]:
SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
TRAIN_COLUMNS = [*SEARCH_COLUMNS, "item_id"]
ITEM_COLUMNS = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text", "item_category_id"]

load_started = time.perf_counter()
train_pairs = pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLUMNS)
benchmark_queries = pd.read_parquet(BENCHMARK_QUERIES_PATH, columns=SEARCH_COLUMNS)
benchmark_items = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=ITEM_COLUMNS)
load_seconds = time.perf_counter() - load_started


def canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame[SEARCH_COLUMNS].copy()
    for column in ("search_query", "search_infm_params_text"):
        result[column] = (
            result[column].astype("string").fillna("<NA>").str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
        )
    for column in ("search_location_id", "search_is_delivery_search", "search_category"):
        result[column] = result[column].astype("string").fillna("<NA>")
    return result


train_canonical = canonical_query_frame(train_pairs)
benchmark_canonical = canonical_query_frame(benchmark_queries)
all_contexts = pd.concat([train_canonical, benchmark_canonical], ignore_index=True)
all_group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)
train_pairs["query_group"] = all_group_ids[: len(train_pairs)]
benchmark_queries["query_group"] = all_group_ids[len(train_pairs) :]
train_pairs["query_text_norm"] = train_canonical["search_query"].astype(str).to_numpy()
train_pairs["category_key"] = train_canonical["search_category"].astype(str).to_numpy()
train_pairs["text_category_key"] = list(zip(train_pairs["query_text_norm"], train_pairs["category_key"], strict=True))
benchmark_queries["query_text_norm"] = benchmark_canonical["search_query"].astype(str).to_numpy()
benchmark_queries["category_key"] = benchmark_canonical["search_category"].astype(str).to_numpy()
benchmark_queries["text_category_key"] = list(zip(benchmark_queries["query_text_norm"], benchmark_queries["category_key"], strict=True))

benchmark_item_ids = benchmark_items["item_id"].astype(str).to_numpy()
benchmark_item_id_set = set(benchmark_item_ids)
proxy_pairs = train_pairs.loc[train_pairs["item_id"].astype(str).isin(benchmark_item_id_set)].copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, valid_idx = next(splitter.split(proxy_pairs, groups=proxy_pairs["query_group"]))
proxy_train_pairs = proxy_pairs.iloc[train_idx].copy()
proxy_valid_pairs = proxy_pairs.iloc[valid_idx].copy()
validation_groups = set(proxy_valid_pairs["query_group"])
assert set(proxy_train_pairs["query_group"]).isdisjoint(validation_groups)

validation_queries = (
    proxy_valid_pairs.sort_values("query_group")
    .drop_duplicates("query_group")
    [["query_group", *SEARCH_COLUMNS, "query_text_norm", "category_key", "text_category_key"]]
    .reset_index(drop=True)
)
gold_by_group = proxy_valid_pairs.groupby("query_group")["item_id"].agg(lambda values: frozenset(values.astype(str))).to_dict()
gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]

# The final inference history may use every train group, but validation may not use its held-out groups.
history_pairs = train_pairs.loc[
    (~train_pairs["query_group"].isin(validation_groups))
    & train_pairs["item_id"].astype(str).isin(benchmark_item_id_set)
].copy()
assert history_pairs["query_group"].isin(validation_groups).sum() == 0

print({
    "load_seconds": round(load_seconds, 2),
    "history_positive_rows": len(history_pairs),
    "validation_positive_rows": len(proxy_valid_pairs),
    "validation_query_groups": len(validation_queries),
})

{'load_seconds': 1.24, 'history_positive_rows': 26377, 'validation_positive_rows': 6633, 'validation_query_groups': 5310}


In [4]:
# Apply the M0 category rule to every propagated historical candidate.
category_to_item_ids = {
    str(category): set(group["item_id"].astype(str))
    for category, group in benchmark_items.groupby("item_category_id", sort=False)
}


def allowed_item_ids(category_key: str) -> set[str]:
    # Category 0 has no corpus partition and is treated as the documented fallback case.
    return category_to_item_ids.get(str(category_key), benchmark_item_id_set)


def filter_by_category(item_ids: list[str], category_key: str, top_k: int = TOP_K) -> list[str]:
    allowed = allowed_item_ids(category_key)
    return [item_id for item_id in item_ids if item_id in allowed][:top_k]


def build_ranked_lookup(frame: pd.DataFrame, key_column: str) -> dict[object, list[str]]:
    """Rank candidates by the number of distinct historical query groups that selected them."""
    # For exact history, query_group itself is the key; do not select it twice.
    columns = [key_column, "item_id"] if key_column == "query_group" else ["query_group", key_column, "item_id"]
    unique_pairs = frame[columns].drop_duplicates()
    counts = unique_pairs.groupby([key_column, "item_id"], sort=False).size().rename("support").reset_index()
    lookup: dict[object, list[str]] = {}
    for key, part in counts.groupby(key_column, sort=False):
        ranked = part.sort_values(["support", "item_id"], ascending=[False, True])
        lookup[key] = ranked["item_id"].astype(str).tolist()
    return lookup


def rankings_from_lookup(lookup: dict[object, list[str]], keys: list[object], categories: list[str]) -> list[list[str]]:
    return [
        filter_by_category(lookup.get(key, []), category)
        for key, category in zip(keys, categories, strict=True)
    ]


def macro_recall(rankings: list[list[str]], gold: list[frozenset[str]], k: int) -> float:
    return float(np.mean([len(set(prediction[:k]) & relevant) / len(relevant) for prediction, relevant in zip(rankings, gold, strict=True)]))


def hit_rate(rankings: list[list[str]], gold: list[frozenset[str]], k: int) -> float:
    return float(np.mean([bool(set(prediction[:k]) & relevant) for prediction, relevant in zip(rankings, gold, strict=True)]))


def source_record(name: str, rankings: list[list[str]], seconds: float, kind: str) -> dict[str, object]:
    record: dict[str, object] = {
        "source": name,
        "kind": kind,
        "coverage": float(np.mean([bool(prediction) for prediction in rankings])),
        "mean_candidates": float(np.mean([len(prediction) for prediction in rankings])),
        "runtime_seconds": seconds,
    }
    for k in METRIC_KS:
        record[f"recall@{k}"] = macro_recall(rankings, gold_sets, k)
    record["hit_rate@50"] = hit_rate(rankings, gold_sets, FINAL_K)
    return record

print({"category_partitions": len(category_to_item_ids), "benchmark_items": len(benchmark_item_ids)})

{'category_partitions': 47, 'benchmark_items': 189212}


## 3. E04a — Exact full-context history

На group-disjoint validation split exact `query_group` lookup обязан иметь нулевое coverage: иначе это был бы leakage. Поэтому следующие ячейки показывают это свойство и отдельно считают, сколько финальных benchmark queries имеют valid exact history в полном train.

In [5]:
exact_validation_lookup = build_ranked_lookup(history_pairs, "query_group")
exact_validation_rankings = rankings_from_lookup(
    exact_validation_lookup,
    validation_queries["query_group"].tolist(),
    validation_queries["category_key"].tolist(),
)
exact_validation_record = source_record("E04a_exact_full_context__validation", exact_validation_rankings, 0.0, "exact")
assert exact_validation_record["coverage"] == 0.0, "Exact lookup must be empty under a group-disjoint split."

# Final-use coverage uses all known train positives that survive in benchmark_items.
exact_final_lookup = build_ranked_lookup(proxy_pairs, "query_group")
exact_benchmark_rankings = rankings_from_lookup(
    exact_final_lookup,
    benchmark_queries["query_group"].tolist(),
    benchmark_queries["category_key"].tolist(),
)
exact_final_coverage = float(np.mean([bool(prediction) for prediction in exact_benchmark_rankings]))
exact_final_mean_candidates = float(np.mean([len(prediction) for prediction in exact_benchmark_rankings]))

exact_coverage_table = pd.DataFrame([{
    "source": "exact normalized full context",
    "validation_coverage": exact_validation_record["coverage"],
    "benchmark_coverage": exact_final_coverage,
    "benchmark_mean_candidates": exact_final_mean_candidates,
}])
display(exact_coverage_table)
print("Zero validation coverage is expected and confirms that validation labels cannot leak into exact history.")

,source,validation_coverage,benchmark_coverage,benchmark_mean_candidates
0,exact normalized full context,0.0,0.024878,0.030995


Zero validation coverage is expected and confirms that validation labels cannot leak into exact history.


## 4. E04b — Relaxed exact history

Следующие ячейки агрегируют benchmark-valid items из других historical query groups. Сначала ключ сохраняет category, затем остаётся только нормализованный query text. Оба источника оцениваются на held-out groups.

In [6]:
history_text_category_lookup = build_ranked_lookup(history_pairs, "text_category_key")
history_text_lookup = build_ranked_lookup(history_pairs, "query_text_norm")

validation_text_category_rankings = rankings_from_lookup(
    history_text_category_lookup,
    validation_queries["text_category_key"].tolist(),
    validation_queries["category_key"].tolist(),
)
validation_text_rankings = rankings_from_lookup(
    history_text_lookup,
    validation_queries["query_text_norm"].tolist(),
    validation_queries["category_key"].tolist(),
)

history_rankings = {
    "E04b_text_category": validation_text_category_rankings,
    "E04b_text_only": validation_text_rankings,
}
history_records = [
    source_record("E04b_text_category", validation_text_category_rankings, 0.0, "relaxed_exact"),
    source_record("E04b_text_only", validation_text_rankings, 0.0, "relaxed_exact"),
]
display(pd.DataFrame(history_records))

# Coverage of the same rules when every train positive is available at final inference.
final_text_category_lookup = build_ranked_lookup(proxy_pairs, "text_category_key")
final_text_lookup = build_ranked_lookup(proxy_pairs, "query_text_norm")
benchmark_relaxed_coverage = pd.DataFrame([
    {
        "source": "text + category",
        "benchmark_coverage": np.mean([bool(prediction) for prediction in rankings_from_lookup(final_text_category_lookup, benchmark_queries["text_category_key"].tolist(), benchmark_queries["category_key"].tolist())]),
    },
    {
        "source": "text only",
        "benchmark_coverage": np.mean([bool(prediction) for prediction in rankings_from_lookup(final_text_lookup, benchmark_queries["query_text_norm"].tolist(), benchmark_queries["category_key"].tolist())]),
    },
])
display(benchmark_relaxed_coverage)

,source,kind,coverage,mean_candidates,runtime_seconds,recall@1,recall@5,recall@10,recall@20,recall@50,recall@200,hit_rate@50
0,E04b_text_category,relaxed_exact,0.647269,13.629567,0.0,0.01882,0.040397,0.055616,0.068981,0.080295,0.087156,0.090584
1,E04b_text_only,relaxed_exact,0.647269,13.629567,0.0,0.01882,0.040397,0.055616,0.068981,0.080295,0.087156,0.090584


,source,benchmark_coverage
0,text + category,0.144372
1,text only,0.147635


## 5. E04c — Nearest historical query

Здесь char-TFIDF ищет самый похожий historical query text и переносит его benchmark-valid positives. Отдельно оценивается вариант только для query text, которых нет в relaxed exact history: это отделяет nearest-neighbour сигнал от простого повторного текста.

In [7]:
NON_WORD_RE = re.compile(r"[^0-9a-zа-я]+")


def normalize_text(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    return " ".join(NON_WORD_RE.sub(" ", text.lower().replace("ё", "е")).split())


nearest_fit_started = time.perf_counter()
history_texts = np.asarray(sorted(history_text_lookup), dtype=object)
nearest_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    preprocessor=normalize_text,
    lowercase=False,
    ngram_range=(3, 5),
    min_df=1,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
    dtype=np.float32,
)
history_query_matrix = nearest_vectorizer.fit_transform(history_texts)
nearest_index_seconds = time.perf_counter() - nearest_fit_started


def nearest_history_rankings(queries: list[str], categories: list[str], *, unseen_only: bool) -> tuple[list[list[str]], list[float], float]:
    started = time.perf_counter()
    rankings, similarities = [], []
    for query, category in zip(queries, categories, strict=True):
        if unseen_only and query in history_text_lookup:
            rankings.append([])
            similarities.append(float("nan"))
            continue
        score = (nearest_vectorizer.transform([query]) @ history_query_matrix.T).toarray().ravel()
        best_idx = int(np.argmax(score))
        best_score = float(score[best_idx])
        if best_score <= 0:
            rankings.append([])
        else:
            rankings.append(filter_by_category(history_text_lookup[str(history_texts[best_idx])], category))
        similarities.append(best_score)
    return rankings, similarities, time.perf_counter() - started

nearest_all_rankings, nearest_all_similarity, nearest_all_seconds = nearest_history_rankings(
    validation_queries["query_text_norm"].tolist(), validation_queries["category_key"].tolist(), unseen_only=False
)
nearest_unseen_rankings, nearest_unseen_similarity, nearest_unseen_seconds = nearest_history_rankings(
    validation_queries["query_text_norm"].tolist(), validation_queries["category_key"].tolist(), unseen_only=True
)
history_rankings["E04c_nearest_char_all"] = nearest_all_rankings
history_rankings["E04c_nearest_char_unseen_text"] = nearest_unseen_rankings
history_records.extend([
    source_record("E04c_nearest_char_all", nearest_all_rankings, nearest_index_seconds + nearest_all_seconds, "nearest"),
    source_record("E04c_nearest_char_unseen_text", nearest_unseen_rankings, nearest_unseen_seconds, "nearest"),
])
nearest_summary = pd.DataFrame(history_records).query("kind == 'nearest'").copy()
nearest_summary["mean_positive_similarity"] = [
    float(np.nanmean(nearest_all_similarity)),
    float(np.nanmean(nearest_unseen_similarity)),
]
display(nearest_summary)

,source,kind,coverage,mean_candidates,runtime_seconds,recall@1,recall@5,recall@10,recall@20,recall@50,recall@200,hit_rate@50,mean_positive_similarity
2,E04c_nearest_char_all,nearest,0.999812,16.120527,9.862798,0.032285,0.059768,0.076775,0.093059,0.106364,0.114920,0.117137,0.901865
3,E04c_nearest_char_unseen_text,nearest,0.352542,2.597928,3.617523,0.013465,0.019586,0.021563,0.024482,0.026554,0.028249,0.027119,0.721784


## 6. Candidate-budget test against retained M1 source

History candidates must not simply increase recall on repeated queries while displacing stronger lexical candidates elsewhere. Следующие ячейки воспроизводят retained M1 E03c, а затем выделяют history source фиксированную квоту в 50-кандидатном списке.

In [8]:
TOKEN_PATTERN = r"(?u)\b[0-9a-zа-я]{2,}\b"


class SparseBM25:
    """Exact Okapi BM25 with a sparse CSC representation, verified in M1."""

    def __init__(self, k1: float = BM25_K1, b: float = BM25_B, epsilon: float = 0.25):
        self.k1, self.b, self.epsilon = k1, b, epsilon

    def fit(self, documents: list[str]) -> "SparseBM25":
        self.vectorizer = CountVectorizer(preprocessor=normalize_text, token_pattern=TOKEN_PATTERN, lowercase=False, dtype=np.float32)
        csr = self.vectorizer.fit_transform(documents)
        self.doc_len = np.asarray(csr.sum(axis=1)).ravel().astype(np.float32)
        self.n_docs = csr.shape[0]
        self.matrix = csr.tocsc()
        del csr
        df = np.diff(self.matrix.indptr).astype(np.float64)
        idf = np.log((self.n_docs - df + 0.5) / (df + 0.5))
        idf[idf < 0] = self.epsilon * float(idf.mean())
        self.idf = idf.astype(np.float32)
        self.norm = self.k1 * (1 - self.b + self.b * self.doc_len / self.doc_len.mean())
        self.analyzer = self.vectorizer.build_analyzer()
        return self

    def top_k(self, query: str, allowed_indices: np.ndarray, k: int = TOP_K) -> np.ndarray:
        scores = np.zeros(self.n_docs, dtype=np.float32)
        for token, query_tf in Counter(self.analyzer(query)).items():
            feature = self.vectorizer.vocabulary_.get(token)
            if feature is None:
                continue
            start, stop = self.matrix.indptr[feature : feature + 2]
            rows, term_tf = self.matrix.indices[start:stop], self.matrix.data[start:stop]
            scores[rows] += query_tf * self.idf[feature] * term_tf * (self.k1 + 1) / (term_tf + self.norm[rows])
        k = min(k, len(allowed_indices))
        allowed_scores = scores[allowed_indices]
        chosen = np.argpartition(allowed_scores, len(allowed_indices) - k)[len(allowed_indices) - k :]
        return allowed_indices[chosen[np.argsort(allowed_scores[chosen])[::-1]]]


def compose_documents(frame: pd.DataFrame, fields: tuple[str, ...]) -> list[str]:
    text = frame[fields[0]].fillna("").astype(str)
    for field in fields[1:]:
        text = text.str.cat(frame[field].fillna("").astype(str), sep=" ")
    return text.tolist()


def rrf(left: np.ndarray, right: np.ndarray, top_k: int = TOP_K, rrf_k: int = 60) -> list[int]:
    scores: dict[int, float] = {}
    for ranked in (left, right):
        for rank, item_idx in enumerate(ranked, start=1):
            scores[int(item_idx)] = scores.get(int(item_idx), 0.0) + 1.0 / (rrf_k + rank)
    return sorted(scores, key=lambda item_idx: (-scores[item_idx], item_idx))[:top_k]


all_item_indices = np.arange(len(benchmark_items), dtype=np.int64)
category_to_indices = {
    str(category): group.index.to_numpy(dtype=np.int64)
    for category, group in benchmark_items.groupby("item_category_id", sort=False)
}
allowed_indices_by_query = [category_to_indices.get(category, all_item_indices) for category in validation_queries["category_key"]]
query_texts = validation_queries["search_query"].fillna("").astype(str).tolist()

m1_started = time.perf_counter()
full_bm25 = SparseBM25().fit(compose_documents(benchmark_items, ("item_title_raw", "item_infm_params_text", "item_description_raw")))
full_bm25_rankings = [full_bm25.top_k(query, allowed) for query, allowed in zip(query_texts, allowed_indices_by_query, strict=True)]

char_vectorizer = TfidfVectorizer(analyzer="char_wb", preprocessor=normalize_text, lowercase=False, ngram_range=(3, 5), min_df=2, max_features=CHAR_MAX_FEATURES, sublinear_tf=True, dtype=np.float32)
char_matrix = char_vectorizer.fit_transform(compose_documents(benchmark_items, ("item_title_raw",)))
char_rankings = []
for query, allowed in zip(query_texts, allowed_indices_by_query, strict=True):
    scores = (char_vectorizer.transform([query]) @ char_matrix.T).toarray().ravel()
    selected = np.argpartition(scores[allowed], len(allowed) - TOP_K)[len(allowed) - TOP_K :]
    char_rankings.append(allowed[selected[np.argsort(scores[allowed][selected])[::-1]]])

m1_rankings = [rrf(bm25, char) for bm25, char in zip(full_bm25_rankings, char_rankings, strict=True)]
m1_predictions = [[str(benchmark_item_ids[item_idx]) for item_idx in ranking] for ranking in m1_rankings]
m1_rebuild_seconds = time.perf_counter() - m1_started
m1_control_record = source_record("M1_E03c_control", m1_predictions, m1_rebuild_seconds, "lexical_control")
assert abs(m1_control_record["recall@50"] - 0.2972636684303351) < 1e-12
print({
    "m1_control_recall@50": m1_control_record["recall@50"],
    "m1_control_runtime_seconds": round(m1_rebuild_seconds, 2),
    "peak_rss_mb": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 * 1024 if sys.platform == "darwin" else 1024), 2),
})

ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


{'m1_control_recall@50': 0.2972636684303351, 'm1_control_runtime_seconds': 386.27, 'peak_rss_mb': 2725.73}


In [9]:
def history_first(history: list[str], lexical: list[str], quota: int, final_k: int = FINAL_K) -> list[str]:
    result: list[str] = []
    for item_id in history[:quota]:
        if item_id not in result:
            result.append(item_id)
    for item_id in lexical:
        if item_id not in result:
            result.append(item_id)
        if len(result) == final_k:
            break
    return result[:final_k]


history_frame = pd.DataFrame(history_records).sort_values("recall@50", ascending=False).reset_index(drop=True)
display(history_frame)

hybrid_records = []
for source_name, rankings in history_rankings.items():
    for quota in HISTORY_QUOTAS:
        predictions = [history_first(history, lexical, quota) for history, lexical in zip(rankings, m1_predictions, strict=True)]
        hybrid_records.append({
            "source": source_name,
            "history_quota": quota,
            "recall@50": macro_recall(predictions, gold_sets, FINAL_K),
            "hit_rate@50": hit_rate(predictions, gold_sets, FINAL_K),
        })
hybrid_frame = pd.DataFrame(hybrid_records).sort_values("recall@50", ascending=False).reset_index(drop=True)
display(hybrid_frame)

best_history_hybrid = hybrid_frame.iloc[0].to_dict()
relaxed_improves = float(best_history_hybrid["recall@50"]) > float(m1_control_record["recall@50"])
print({
    "lexical_control_recall@50": round(float(m1_control_record["recall@50"]), 6),
    "best_history_hybrid": best_history_hybrid,
    "relaxed_history_improves_overall": relaxed_improves,
})

,source,kind,coverage,mean_candidates,runtime_seconds,recall@1,recall@5,recall@10,recall@20,recall@50,recall@200,hit_rate@50
0,E04c_nearest_char_all,nearest,0.999812,16.120527,9.862798,0.032285,0.059768,0.076775,0.093059,0.106364,0.114920,0.117137
1,E04b_text_category,relaxed_exact,0.647269,13.629567,0.000000,0.018820,0.040397,0.055616,0.068981,0.080295,0.087156,0.090584
2,E04b_text_only,relaxed_exact,0.647269,13.629567,0.000000,0.018820,0.040397,0.055616,0.068981,0.080295,0.087156,0.090584
3,E04c_nearest_char_unseen_text,nearest,0.352542,2.597928,3.617523,0.013465,0.019586,0.021563,0.024482,0.026554,0.028249,0.027119


,source,history_quota,recall@50,hit_rate@50
0,E04c_nearest_char_all,20,0.333928,0.347081
1,E04b_text_category,20,0.328211,0.341431
2,E04b_text_only,20,0.328211,0.341431
3,E04c_nearest_char_all,10,0.324376,0.337288
4,E04b_text_category,10,0.320354,0.333333
5,E04b_text_only,10,0.320354,0.333333
6,E04c_nearest_char_all,5,0.316220,0.328625
7,E04b_text_category,5,0.312480,0.324859
8,E04b_text_only,5,0.312480,0.324859
9,E04c_nearest_char_all,3,0.312342,0.324294


{'lexical_control_recall@50': 0.297264, 'best_history_hybrid': {'source': 'E04c_nearest_char_all', 'history_quota': 20, 'recall@50': 0.33392765970167104, 'hit_rate@50': 0.3470809792843691}, 'relaxed_history_improves_overall': True}


## 7. Confirmation: второй group-disjoint split

Первичный выигрыш слишком велик, чтобы принять его на одном seed. Следующие ячейки повторяют **только выбранную policy** — nearest historical query с quota 20 — на `GroupShuffleSplit(random_state=314)`. Они переиспользуют готовые M1 индексы, но строят историю исключительно вне второго validation fold.

In [10]:
SECONDARY_SEED = 314
SECONDARY_QUOTA = int(best_history_hybrid["history_quota"])
assert best_history_hybrid["source"] == "E04c_nearest_char_all"

second_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SECONDARY_SEED)
_, secondary_valid_idx = next(second_splitter.split(proxy_pairs, groups=proxy_pairs["query_group"]))
secondary_valid_pairs = proxy_pairs.iloc[secondary_valid_idx].copy()
secondary_groups = set(secondary_valid_pairs["query_group"])
secondary_history_pairs = train_pairs.loc[
    (~train_pairs["query_group"].isin(secondary_groups))
    & train_pairs["item_id"].astype(str).isin(benchmark_item_id_set)
].copy()
assert secondary_history_pairs["query_group"].isin(secondary_groups).sum() == 0

secondary_queries = (
    secondary_valid_pairs.sort_values("query_group")
    .drop_duplicates("query_group")
    [["query_group", "search_query", "query_text_norm", "category_key"]]
    .reset_index(drop=True)
)
secondary_gold_by_group = secondary_valid_pairs.groupby("query_group")["item_id"].agg(lambda values: frozenset(values.astype(str))).to_dict()
secondary_gold = [secondary_gold_by_group[group] for group in secondary_queries["query_group"]]
secondary_history_text_lookup = build_ranked_lookup(secondary_history_pairs, "query_text_norm")

confirmation_started = time.perf_counter()
secondary_history_texts = np.asarray(sorted(secondary_history_text_lookup), dtype=object)
secondary_nearest_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    preprocessor=normalize_text,
    lowercase=False,
    ngram_range=(3, 5),
    min_df=1,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
    dtype=np.float32,
)
secondary_history_matrix = secondary_nearest_vectorizer.fit_transform(secondary_history_texts)

secondary_nearest_rankings = []
secondary_m1_predictions = []
for query, category_key in zip(secondary_queries["query_text_norm"], secondary_queries["category_key"], strict=True):
    # Historical source: nearest non-validation query text, then category-filtered positives.
    history_scores = (secondary_nearest_vectorizer.transform([query]) @ secondary_history_matrix.T).toarray().ravel()
    neighbor = int(np.argmax(history_scores))
    if float(history_scores[neighbor]) > 0:
        secondary_nearest_rankings.append(filter_by_category(secondary_history_text_lookup[str(secondary_history_texts[neighbor])], category_key))
    else:
        secondary_nearest_rankings.append([])

    # Reuse the M1 BM25 and title-char-TFIDF indices; only retrieve for the new fold.
    allowed = category_to_indices.get(category_key, all_item_indices)
    bm25_ranking = full_bm25.top_k(str(query), allowed)
    char_scores = (char_vectorizer.transform([str(query)]) @ char_matrix.T).toarray().ravel()
    char_selected = np.argpartition(char_scores[allowed], len(allowed) - TOP_K)[len(allowed) - TOP_K :]
    char_ranking = allowed[char_selected[np.argsort(char_scores[allowed][char_selected])[::-1]]]
    secondary_m1_predictions.append([str(benchmark_item_ids[item_idx]) for item_idx in rrf(bm25_ranking, char_ranking)])

secondary_hybrid_predictions = [
    history_first(history, lexical, SECONDARY_QUOTA)
    for history, lexical in zip(secondary_nearest_rankings, secondary_m1_predictions, strict=True)
]

def macro_recall_for(rankings: list[list[str]], gold: list[frozenset[str]], k: int = FINAL_K) -> float:
    return float(np.mean([len(set(prediction[:k]) & relevant) / len(relevant) for prediction, relevant in zip(rankings, gold, strict=True)]))

secondary_record = {
    "seed": SECONDARY_SEED,
    "query_groups": len(secondary_queries),
    "history_coverage": float(np.mean([bool(prediction) for prediction in secondary_nearest_rankings])),
    "history_quota": SECONDARY_QUOTA,
    "m1_control_recall@50": macro_recall_for(secondary_m1_predictions, secondary_gold),
    "history_hybrid_recall@50": macro_recall_for(secondary_hybrid_predictions, secondary_gold),
    "runtime_seconds": time.perf_counter() - confirmation_started,
}
secondary_record["absolute_gain"] = secondary_record["history_hybrid_recall@50"] - secondary_record["m1_control_recall@50"]
secondary_record["improves"] = secondary_record["absolute_gain"] > 0
display(pd.DataFrame([secondary_record]))

,seed,query_groups,history_coverage,history_quota,m1_control_recall@50,history_hybrid_recall@50,runtime_seconds,absolute_gain,improves
0,314,5310,1.0,20,0.306026,0.338708,323.106823,0.032682,True


## 8. Results, ClearML logging и решение

В следующих ячейках объединяются primary и confirmation split. Exact full-context history рассматривается отдельно: его validation coverage нулевое по конструкции group-disjoint split, поэтому решение опирается на benchmark coverage. Relaxed/nearest policy принимается только при положительном общем gain на обоих seeds.

In [11]:
all_records = [exact_validation_record, *history_records, m1_control_record]
for record in all_records:
    series = str(record["source"])
    clearml_logger.report_scalar("Recall@50", series, float(record["recall@50"]), 0)
    clearml_logger.report_scalar("Coverage", series, float(record["coverage"]), 0)
    clearml_logger.report_scalar("Runtime seconds", series, float(record["runtime_seconds"]), 0)
for _, row in hybrid_frame.iterrows():
    series = f'{row["source"]}__quota_{int(row["history_quota"])}'
    clearml_logger.report_scalar("Hybrid Recall@50", series, float(row["recall@50"]), 0)
    clearml_logger.report_scalar("Hybrid HitRate@50", series, float(row["hit_rate@50"]), 0)
clearml_logger.report_scalar("Secondary M1 control Recall@50", "seed_314", float(secondary_record["m1_control_recall@50"]), 0)
clearml_logger.report_scalar("Secondary history hybrid Recall@50", "seed_314", float(secondary_record["history_hybrid_recall@50"]), 0)
clearml_logger.report_scalar("Secondary absolute gain", "seed_314", float(secondary_record["absolute_gain"]), 0)
clearml_logger.report_scalar("Final benchmark exact-history coverage", "exact_full_context", exact_final_coverage, 0)
clearml_logger.report_scalar("M2 notebook seconds", "runtime", time.perf_counter() - NOTEBOOK_STARTED, 0)
try:
    clearml_logger.report_table("M2 history source summary", "sources", 0, table_plot=history_frame)
    clearml_logger.report_table("M2 candidate-budget summary", "primary_hybrids", 0, table_plot=hybrid_frame)
    clearml_logger.report_table("M2 confirmation summary", "seed_314", 0, table_plot=pd.DataFrame([secondary_record]))
    clearml_table_status = "logged"
except Exception as exc:
    clearml_table_status = f"table logging skipped: {type(exc).__name__}"

confirmed_relaxed_improvement = relaxed_improves and bool(secondary_record["improves"])
if confirmed_relaxed_improvement:
    relaxed_decision = f"retain E04c_nearest_char_all with quota={SECONDARY_QUOTA}; positive gain on seeds 42 and 314"
else:
    relaxed_decision = "reject relaxed and nearest sources: improvement did not reproduce on both group-disjoint seeds"
exact_decision = (
    "retain exact full-context history as a candidate-pool source before M5/M6 selection"
    if exact_final_coverage > 0 else "reject exact full-context history: no final benchmark coverage"
)

clearml_task.set_parameter("results/exact_final_coverage", exact_final_coverage)
clearml_task.set_parameter("results/primary_best_hybrid_recall_at_50", float(best_history_hybrid["recall@50"]))
clearml_task.set_parameter("results/secondary_history_hybrid_recall_at_50", float(secondary_record["history_hybrid_recall@50"]))
clearml_task.set_parameter("results/relaxed_decision", relaxed_decision)
clearml_task.set_parameter("results/exact_decision", exact_decision)
clearml_task.close()

print({
    "exact_history_decision": exact_decision,
    "relaxed_history_decision": relaxed_decision,
    "secondary_record": secondary_record,
    "clearml_table_status": clearml_table_status,
    "notebook_seconds": round(time.perf_counter() - NOTEBOOK_STARTED, 2),
})

ClearML Monitor: Reporting detected, reverting back to iteration based reporting


{'exact_history_decision': 'retain exact full-context history as a candidate-pool source before M5/M6 selection', 'relaxed_history_decision': 'retain E04c_nearest_char_all with quota=20; positive gain on seeds 42 and 314', 'secondary_record': {'seed': 314, 'query_groups': 5310, 'history_coverage': 1.0, 'history_quota': 20, 'm1_control_recall@50': 0.30602606642154667, 'history_hybrid_recall@50': 0.33870781394792693, 'runtime_seconds': 323.10682279200023, 'absolute_gain': 0.032681747526380256, 'improves': True}, 'clearml_table_status': 'logged', 'notebook_seconds': 783.73}


## M2 conclusion

The executed decision cell is the source of truth. Exact history adds candidates only through a generic normalized search context; it never uses `query_id`. The relaxed/nearest history policy is retained only if it improves macro Recall@50 with the same quota on the primary seed and independent seed 314. M5–M6 will later decide the final union and selector across all accepted sources.